# 13 — Challenge solution

Debrief only. Same kernel as the student notebook, after `one`, `invoices`, `people`, `model` and `Runner` exist.

Both designs get Puja right: **6**, **$36.64**, **Jane Peacock**. That is the control. The finding is the bill.

The router pays for the same three facts twice over: the desk calls a specialist, the specialist runs its own model turn, and the desk then reads the specialist's answer back into its own context. Four extra model calls for an answer one agent already had.

In [8]:
puja = (
    "How many invoices does Puja Srivastava have, what did she spend, "
    "and who is her support rep?"
)

# 1. the single agent, three tools -- already built in Part 1
one_puja = await Runner.run(one, puja)
one_text = one_puja.final_output
one_tokens = one_puja.context_wrapper.usage.total_tokens
one_requests = one_puja.context_wrapper.usage.requests

# 2. the same two specialists, sharp descriptions, allowed to use both
router = Agent(
    name="Desk",
    instructions="Use your specialists. Do not invent numbers or names.",
    model=model,
    tools=[
        invoices.as_tool(
            tool_name="ask_invoices",
            tool_description="Invoice count and total spend only. Cannot name a support representative.",
        ),
        people.as_tool(
            tool_name="ask_people",
            tool_description="Support representative name only. Cannot count invoices or spend.",
        ),
    ],
)
router_puja = await Runner.run(router, puja)
router_text = router_puja.final_output
router_tokens = router_puja.context_wrapper.usage.total_tokens
router_requests = router_puja.context_wrapper.usage.requests

print("one agent:", one_text)
print()
print("router:   ", router_text)

one agent: Puja Srivastava has **6** invoices and has spent **$36.64**. Her support rep is **Jane Peacock**.

router:    - **Invoices:** 6  
- **Total spend:** 36.64  
- **Support rep:** Jane Peacock


In [9]:
def facts_in(text):
    t = str(text)
    return {
        "count": bool(re.search(r"\b6\b", t)),
        "spend": "36.64" in t,
        "rep": "jane" in t.lower() or "peacock" in t.lower(),
    }


one_facts = facts_in(one_text)
router_facts = facts_in(router_text)

# The single agent is the baseline. One process, three tools -- it should not miss.
assert all(one_facts.values()), f"one agent should have all three facts: {one_facts}"
assert router_text and str(router_text).strip(), "the router should still produce an answer"

# The finding is structural. It does not depend on the model's mood.
assert router_requests > one_requests, "the router design should make more model calls"
assert router_tokens > one_tokens, "the router design should send more tokens for the same job"

print(f"one agent: {one_requests} requests, {one_tokens} tokens, facts {one_facts}")
print(f"router:    {router_requests} requests, {router_tokens} tokens, facts {router_facts}")
print(f"the second agent cost {router_tokens / one_tokens:.1f}x the tokens for the same answer")

if not all(router_facts.values()):
    missing = [k for k, v in router_facts.items() if not v]
    print()
    print("NOTE: the router dropped", missing, "- and that is not a broken cell.")
    print("A topology that hands work across boxes can lose a fact on the way back.")
    print("You watched the same failure in module 08 Part 3, with handoffs.")
    print("It is one more cost of the second agent, on top of the tokens.")

one agent: 2 requests, 561 tokens, facts {'count': True, 'spend': True, 'rep': True}
router:    6 requests, 1226 tokens, facts {'count': True, 'spend': True, 'rep': True}
the second agent cost 2.2x the tokens for the same answer
